# LoopTor — Colab pipeline (Qwen Local Hackathon 2026)
**Run order:** Runtime → Run all. Everything saves to Google Drive (`MyDrive/looptor/`) so a disconnect loses nothing.
Steps: base eval (before) → Best-of-N data → dataset → smoke train → full train → tuned eval (after) → GGUF export.
Set your Hugging Face **Write** token in Colab Secrets as `HF_TOKEN` before running (only needed for the upload cell).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/looptor'
for d in ['results/base', 'results/tuned', 'data', 'adapter', 'gguf']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
print('Drive ready:', BASE)

In [ ]:
# Repo (benchmark, detector, scoring, dataset builder)
%cd /content
!rm -rf looptor
!git clone -q https://github.com/Daimx1214/looptor.git
%cd /content/looptor
!ls loopbench scripts

In [ ]:
# Install Unsloth + deps
!pip install -q unsloth transformers datasets trl peft accelerate bitsandbytes
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# Load base Qwen3-1.7B (4-bit) once; reused for eval, Best-of-N and training
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-1.7B', max_seq_length=2048, load_in_4bit=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('loaded')

In [ ]:
# Batched generation helper. Returns rows in the same schema as scripts/run_eval.py
import json, time
def batch_generate(prompts, max_new=1024, temp=0.6, top_p=0.95, batch=8):
    rows = []
    for i in range(0, len(prompts), batch):
        chunk = prompts[i:i+batch]
        texts = [tokenizer.apply_chat_template(
            [{'role': 'user', 'content': p}], tokenize=False,
            add_generation_prompt=True, enable_thinking=True) for p in chunk]
        enc = tokenizer(texts, return_tensors='pt', padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                                 temperature=temp, top_p=top_p, top_k=20,
                                 pad_token_id=tokenizer.pad_token_id)
        for j, o in enumerate(out):
            gen = o[enc['input_ids'].shape[1]:]
            text = tokenizer.decode(gen, skip_special_tokens=True)
            think, _, content = text.partition('</think>')
            think = think.replace('<think>', '').strip()
            rows.append({'thinking': think, 'content': content.strip(),
                         'full_text': text, 'tokens': int((gen != tokenizer.pad_token_id).sum()),
                         'done_reason': 'length' if len(gen) >= max_new else 'stop'})
        print(f'{min(i+batch, len(prompts))}/{len(prompts)}', flush=True)
    return rows

In [ ]:
# STEP 1 — BASE EVAL (before numbers) on test + stress, temp 0.6, cap 2048
FastLanguageModel.for_inference(model)
import subprocess
for s in ['test', 'stress']:
    items = [json.loads(l) for l in open(f'loopbench/{s}.jsonl')]
    gens = batch_generate([it['prompt'] for it in items], max_new=2048, temp=0.6)
    with open(f'{BASE}/results/base/{s}.jsonl', 'w') as f:
        for it, g in zip(items, gens):
            f.write(json.dumps({**it, **g, 'model': 'qwen3-1.7b-base'}, ensure_ascii=False) + '\n')
    print(s, 'done')
!cp -r {BASE}/results/base results/base_run
!python scripts/score_loops.py results/base_run --save {BASE}/results/score_base.json

In [ ]:
# STEP 2 — BEST-OF-N generation on the train pool (N=6, temp 0.8). Resumable:
# prompts already present in the Drive file are skipped. Runs in chunks of 100.
RAW = f'{BASE}/data/bestofn_raw.jsonl'
done_ids = set()
if os.path.exists(RAW):
    done_ids = {json.loads(l)['id'] for l in open(RAW)}
items = [json.loads(l) for l in open('loopbench/train_pool.jsonl')]
todo = [it for it in items if it['id'] not in done_ids]
print('prompts left:', len(todo))
for c in range(0, len(todo), 100):
    chunk = todo[c:c+100]
    prompts, meta = [], []
    for it in chunk:
        for s in range(6):
            prompts.append(it['prompt']); meta.append((it, s))
    gens = batch_generate(prompts, max_new=1024, temp=0.8)
    with open(RAW, 'a') as f:
        for (it, s), g in zip(meta, gens):
            f.write(json.dumps({'id': it['id'], 'sample': s, 'category': it['category'],
                                'prompt': it['prompt'], 'gold': it['gold'], 'check': it['check'],
                                **g}, ensure_ascii=False) + '\n')
    print('chunk saved', c + len(chunk), flush=True)
!cp {RAW} data/bestofn_raw.jsonl

In [ ]:
# STEP 3 — Build the 4-part dataset (~3,000) and eyeball samples
!python scripts/build_dataset.py
!cp data/train.jsonl data/stats.json {BASE}/data/
import random
rows = [json.loads(l) for l in open('data/train.jsonl')]
print('total:', len(rows))
for r in random.sample(rows, 3):
    print('---'); print(r['messages'][1]['content'][:280])

In [ ]:
# STEP 4 — SMOKE TRAIN: 200 examples, 1 epoch (format check only)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
rows = [json.loads(l) for l in open('data/train.jsonl')]
def fmt(r):
    return {'text': tokenizer.apply_chat_template(r['messages'], tokenize=False)}
smoke = Dataset.from_list([fmt(r) for r in rows[:200]])
model2 = model
trainer = SFTTrainer(model=model2, tokenizer=tokenizer, train_dataset=smoke,
    args=SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   num_train_epochs=1, learning_rate=2e-4, logging_steps=5,
                   output_dir='/tmp/smoke', report_to='none', max_length=1024))
from unsloth import train_on_responses_only
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n',
                                  response_part='<|im_start|>assistant\n')
trainer.train()
print('smoke done — check the loss went down and samples end with <|im_end|>')

In [ ]:
# STEP 5 — FULL TRAIN: LoRA r16/a32 all-linear, 3 epochs, lr 2e-4 (Guide Ch. 12)
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-1.7B', max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth')
full = Dataset.from_list([fmt(r) for r in rows])
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=full,
    args=SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   num_train_epochs=3, learning_rate=2e-4, logging_steps=10,
                   output_dir='/tmp/full', report_to='none', max_length=1024))
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n',
                                  response_part='<|im_start|>assistant\n')
trainer.train()
model.save_pretrained(f'{BASE}/adapter'); tokenizer.save_pretrained(f'{BASE}/adapter')
print('adapter saved to Drive')

In [ ]:
# STEP 6 — TUNED EVAL (after numbers), same protocol as Step 1
FastLanguageModel.for_inference(model)
for s in ['test', 'stress']:
    items = [json.loads(l) for l in open(f'loopbench/{s}.jsonl')]
    gens = batch_generate([it['prompt'] for it in items], max_new=2048, temp=0.6)
    with open(f'{BASE}/results/tuned/{s}.jsonl', 'w') as f:
        for it, g in zip(items, gens):
            f.write(json.dumps({**it, **g, 'model': 'looptor'}, ensure_ascii=False) + '\n')
!cp -r {BASE}/results/tuned results/tuned_run
!python scripts/score_loops.py results/tuned_run --save {BASE}/results/score_tuned.json
print(open(f'{BASE}/results/score_base.json').read())
print(open(f'{BASE}/results/score_tuned.json').read())

In [ ]:
# STEP 7 — GGUF Q4_K_M export (Guide Ch. 13) -> Drive
model.save_pretrained_gguf(f'{BASE}/gguf', tokenizer, quantization_method='q4_k_m')
!ls -la {BASE}/gguf

In [ ]:
# STEP 8 (optional) — Upload GGUF to Hugging Face (needs HF_TOKEN in Colab Secrets)
from google.colab import userdata
from huggingface_hub import HfApi
tok = userdata.get('HF_TOKEN')
api = HfApi(token=tok)
api.create_repo('Daimx1214/looptor-1.7b', repo_type='model', exist_ok=True)
api.upload_folder(folder_path=f'{BASE}/gguf', repo_id='Daimx1214/looptor-1.7b', repo_type='model')
print('uploaded: https://huggingface.co/Daimx1214/looptor-1.7b')

## After this notebook
1. Download the GGUF from Drive, `ollama create looptor -f Modelfile`, run the 20-prompt match check (Guide Ch. 13).
2. Record the demo video (Ch. 17), publish model card on HF + ModelScope (Ch. 15).
3. Submit: PR + demo post + form (Ch. 16). **Deadline: Sat 28 Nov 2026, 11:59 PM PKT — aim Friday.**